# 문항 2 · 타입 가드와 제네릭 저장소

이 노트북은 첫 번째 노트북을 먼저 실행하지 않아도 동작합니다. 두 문항은 동일한 회원·상품 모델을 사용합니다.

In [ ]:
interface Identifiable {
    id: number;
}

type UserRole = 'admin' | 'customer' | 'guest';
type Category = 'electronics' | 'clothing' | 'books';

interface User extends Identifiable {
    name: string;
    email: string;
    role: UserRole;
    isActive: boolean;
}

interface Product extends Identifiable {
    title: string;
    price: number;
    category: Category;
    stock: number;
    discountRate?: number;
}

const users: User[] = [
    { id: 1, name: '지민', email: 'jimin@example.com', role: 'admin', isActive: true },
    { id: 2, name: '수현', email: 'suhyeon@example.com', role: 'customer', isActive: true },
    { id: 3, name: '민준', email: 'minjun@example.com', role: 'guest', isActive: false },
    { id: 4, name: '하윤', email: 'hayun@example.com', role: 'customer', isActive: false },
];

const products: Product[] = [
    { id: 101, title: '키보드', price: 50000, category: 'electronics', stock: 5, discountRate: 0.2 },
    { id: 102, title: '셔츠', price: 30000, category: 'clothing', stock: 3 },
    { id: 103, title: 'TypeScript 입문', price: 25000, category: 'books', stock: 8, discountRate: 0 },
];
console.log(`회원·상품 샘플 ${users.length + products.length}개 준비`);

## 유니언 타입 좁히기

필수 필드인 email의 존재를 검사합니다. 참 분기에서는 회원 필드를 사용하고, 나머지 분기에서는 상품 필드를 사용합니다.

In [2]:
function renderItemSummary(item: User | Product): string {
    if ('email' in item) {
        return `[회원] ${item.name} (${item.role})`;
    } else {
        return `[상품] ${item.title} - ${item.price}원`;
    }
}

const mixedItems: (User | Product)[] = [...users, ...products];
const summaries: string[] = mixedItems.map(renderItemSummary);
if (summaries[0] !== '[회원] 지민 (admin)' || summaries[4] !== '[상품] 키보드 - 50000원') {
    throw new Error('회원 또는 상품 요약이 예상과 다릅니다.');
}
console.log(summaries.join('\n'));

[회원] 지민 (admin)
[회원] 수현 (customer)
[회원] 민준 (guest)
[회원] 하윤 (customer)
[상품] 키보드 - 50000원
[상품] 셔츠 - 30000원
[상품] TypeScript 입문 - 25000원


## 인메모리 저장소

숫자 ID를 가진 타입을 저장할 수 있습니다. 같은 저장소에서 ID가 중복되면 추가를 거절합니다. 전체 조회는 배열을 얕게 복사하므로 반환 배열에서 항목을 제거해도 저장소의 배열에는 영향이 없습니다.

In [3]:
class InMemoryStorage<T extends Identifiable> {
    private items: T[] = [];

    add(item: T): void {
        if (this.getById(item.id) !== undefined) {
            throw new Error(`이미 존재하는 ID: ${item.id}`);
        }
        this.items.push(item);
    }

    getById(id: number): T | undefined {
        return this.items.find(item => item.id === id);
    }

    getAll(): T[] {
        return [...this.items];
    }
}

console.log('회원과 상품에 공통으로 사용할 저장소를 정의했습니다.');

회원과 상품에 공통으로 사용할 저장소를 정의했습니다.


In [4]:
const userStorage = new InMemoryStorage<User>();
const productStorage = new InMemoryStorage<Product>();
users.forEach(user => userStorage.add(user));
products.forEach(product => productStorage.add(product));

const foundUser = userStorage.getById(1);
const foundProduct = productStorage.getById(101);
const missingUser = userStorage.getById(999);
const userSnapshot = userStorage.getAll();
userSnapshot.pop();

if (foundUser?.email !== 'jimin@example.com' || foundProduct?.title !== '키보드'
    || missingUser !== undefined || userStorage.getAll().length !== users.length) {
    throw new Error('저장소 조회 또는 배열 복사 결과가 예상과 다릅니다.');
}

try {
    userStorage.add(foundUser);
    throw new Error('중복 ID가 추가됐습니다.');
} catch (error: unknown) {
    if (!(error instanceof Error) || error.message !== '이미 존재하는 ID: 1') {
        throw error;
    }
    console.log('중복 ID 거절:', error.message);
}

console.log('회원 조회:', foundUser.email);
console.log('상품 조회:', foundProduct.title);
console.log('없는 ID 조회:', missingUser);
console.log('복사 배열과 원본의 길이:', userSnapshot.length, userStorage.getAll().length);
console.log('전체 상품:', productStorage.getAll().map(product => product.title).join(', '));

중복 ID 거절: 이미 존재하는 ID: 1


회원 조회: jimin@example.com


상품 조회: 키보드


없는 ID 조회: undefined


복사 배열과 원본의 길이: 3 4


전체 상품: 키보드, 셔츠, TypeScript 입문
